In [22]:
import pandas as pd

# 1. Load dataset
df = pd.read_csv("RealEstate_Prices.csv")

# 2. Clean column names
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^\w]+", "_", regex=True)
              .str.strip("_"))

print("Original Data:")
print(df.head())
print("\nShape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

# Ensure numeric columns have numeric data types
cols = ["property_id", "bedrooms", "bathrooms", "area_sq_ft",
        "year_built", "sale_price", "sale_year"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows missing essential information
df = df.dropna(subset=["property_id", "sale_price", "area_sq_ft", "sale_year"])

# 3. Handle remaining missing values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().any() and not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

# Remove duplicate records
df = df.drop_duplicates()

# Standardize text
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

print("\nMissing Values After Cleaning:\n", df.isnull().sum())

# 4. Merge neighborhood information (example data)
neighborhood_data = pd.DataFrame({
    "neighborhood": ["downtown", "suburb", "riverside",
                     "hillview", "lakeside"],
    "avg_income": [75000, 65000, 60000, 70000, 72000]
})

df = pd.merge(df, neighborhood_data, on="neighborhood", how="left")

# 5. Filter properties sold from 2021 to 2023
filtered_data = df[df["sale_year"].between(2021, 2023)]
print("\nProperties Sold from 2021 to 2023:")
print(filtered_data.head())

# 6. Encode categorical columns
df_encoded = pd.get_dummies(
    df, columns=["property_type", "neighborhood"], dtype=int
)

# 7. Aggregate average prices
print("\nAverage Price by Neighborhood:")
print(df.groupby("neighborhood")["sale_price"].mean())

print("\nAverage Price by Property Type:")
print(df.groupby("property_type")["sale_price"].mean())

# Price per square foot
df["price_per_sq_ft"] = df["sale_price"] / df["area_sq_ft"]

print("\nPrice per Square Foot:")
print(df[["property_id", "price_per_sq_ft"]].head())

# 8. Identify outliers using IQR
q1 = df["sale_price"].quantile(0.25)
q3 = df["sale_price"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("\nPotential Sale Price Outliers:")
print(df[~df["sale_price"].between(lower, upper)][
    ["property_id", "sale_price"]
])

# Remove outliers from sale price and area
q1_area = df["area_sq_ft"].quantile(0.25)
q3_area = df["area_sq_ft"].quantile(0.75)
iqr_area = q3_area - q1_area

lower_area = q1_area - 1.5 * iqr_area
upper_area = q3_area + 1.5 * iqr_area

df_clean = df[
    df["sale_price"].between(lower, upper) &
    df["area_sq_ft"].between(lower_area, upper_area)
].copy()

print("\nShape Before Outlier Removal:", df.shape)
print("Shape After Outlier Removal:", df_clean.shape)



Original Data:
   property_id property_type neighborhood  bedrooms  bathrooms  area_sq_ft  \
0         1001         Villa     Hillview         3        2.0        1743   
1         1002     Apartment     Hillview         4        4.0        1592   
2         1003     Apartment    Riverside         1        3.0        3618   
3         1004     Apartment       Suburb         4        3.0        1508   
4         1005         Villa     Hillview         3        2.0        2641   

   year_built  sale_price  sale_year  
0        2018      586261       2022  
1        2011      455540       2023  
2        2003      292714       2021  
3        2002      831899       2023  
4        2011      383876       2023  

Shape: (100, 9)

Missing Values:
 property_id      0
property_type    0
neighborhood     2
bedrooms         0
bathrooms        3
area_sq_ft       0
year_built       0
sale_price       0
sale_year        0
dtype: int64

Missing Values After Cleaning:
 property_id      0
property_ty

In [23]:
import pandas  as pd

df= pd.read_csv("RealEstate_Prices.csv")
print(df.head())

#clean cloumn names
df.columns=(df.columns.str.strip().str.lower().str.replace(r"[^\w]+","_",regex=True).str.strip("_"))

print(df.columns)

print (df.shape)

print(df.isnull().sum())
df=df.drop_duplicates()
df=df.dropna(subset=['area_sq_ft'])

print(df.dtypes)
for col in df.select_dtypes(include="object"):
    df[col]=df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

df['bathrooms']=df["bathrooms"].fillna(df["bathrooms"].median())

df["bathrooms"]=df["bathrooms"].astype(int)

print(df.dtypes)
print(df.isnull().sum())

df = df.drop_duplicates()

# Standardize text
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

for col in df.select_dtypes(include='number'):
   # 1. Skip the ID column and skip binary dummy columns
    if col == 'property_id' or df[col].nunique() <= 2:
        continue
    q1=df[col].quantile(0.25)
    q3=df[col].quantile(0.75)
    iqr=q3-q1

    df[col]=df[col].clip(q1-1.5*iqr,q3+1.5*iqr)

print(df.shape)

df2=pd.DataFrame({
    "neighborhood": ["downtown", "suburb", "riverside",
                     "hillview", "lakeside"],
    "avg_income": [75000, 65000, 60000, 70000, 72000]
})

df=pd.merge(df,df2,on='neighborhood',how='left')
print(df.head())

filtered_data = df[df["sale_year"].between(2021, 2023)]
print("\nProperties Sold from 2021 to 2023:")
print(filtered_data.head())

# 6. Encode categorical columns
df_encoded = pd.get_dummies(
    df, columns=["property_type", "neighborhood"], dtype=int
)

print("avg sales by neighborhood")

print(df.groupby('neighborhood')['sale_price'].mean())

   Property ID Property Type Neighborhood  Bedrooms  Bathrooms  Area (sq ft)  \
0         1001         Villa     Hillview         3        2.0          1743   
1         1002     Apartment     Hillview         4        4.0          1592   
2         1003     Apartment    Riverside         1        3.0          3618   
3         1004     Apartment       Suburb         4        3.0          1508   
4         1005         Villa     Hillview         3        2.0          2641   

   Year Built  Sale Price ($)  Sale Year  
0        2018          586261       2022  
1        2011          455540       2023  
2        2003          292714       2021  
3        2002          831899       2023  
4        2011          383876       2023  
Index(['property_id', 'property_type', 'neighborhood', 'bedrooms', 'bathrooms',
       'area_sq_ft', 'year_built', 'sale_price', 'sale_year'],
      dtype='object')
(100, 9)
property_id      0
property_type    0
neighborhood     2
bedrooms         0
bathrooms  